# 43 — Window Ranking

Ranks candidate 4–7 day windows using the comparison outputs.

In [ ]:
import os, json, hashlib, platform, sys, math, time
from pathlib import Path
from datetime import datetime, timezone
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import requests
import yaml

PROJECT_ROOT = Path.cwd().resolve().parents[0] if (Path.cwd().name == "notebooks") else Path.cwd().resolve()
OUTPUTS = PROJECT_ROOT / "outputs"
CONFIGS = PROJECT_ROOT / "configs"
OUTPUTS.mkdir(exist_ok=True)

def utcnow():
    return datetime.now(timezone.utc).isoformat()

def sha256_file(p: Path) -> str:
    h = hashlib.sha256()
    with p.open("rb") as f:
        for chunk in iter(lambda: f.read(1024*1024), b""):
            h.update(chunk)
    return h.hexdigest()

def write_json(p: Path, obj):
    p.parent.mkdir(parents=True, exist_ok=True)
    p.write_text(json.dumps(obj, indent=2, sort_keys=True), encoding="utf-8")

def load_json(p: Path):
    return json.loads(p.read_text(encoding="utf-8"))

def load_yaml(p: Path):
    return yaml.safe_load(p.read_text(encoding="utf-8")) if p.exists() else {}

def safe_read_csv(path_str):
    p = Path(path_str)
    if not p.is_absolute():
        p = PROJECT_ROOT / p
    if not p.exists():
        return pd.DataFrame(), {"path": str(p), "status": "missing"}
    try:
        if p.stat().st_size == 0:
            return pd.DataFrame(), {"path": str(p), "status": "empty_file"}
        df = pd.read_csv(p)
        if df.empty:
            return pd.DataFrame(), {"path": str(p), "sha256": sha256_file(p), "status": "no_rows"}
        return df, {"path": str(p), "sha256": sha256_file(p), "status": "ok"}
    except pd.errors.EmptyDataError:
        return pd.DataFrame(), {"path": str(p), "status": "empty_data"}
    except Exception as e:
        return pd.DataFrame(), {"path": str(p), "status": f"error:{e}"}

def manifest_base(step_name: str, config_paths=None):
    return {
        "project": "AirQuality26",
        "step": step_name,
        "created_at_utc": utcnow(),
        "platform": {
            "python": sys.version,
            "platform": platform.platform(),
        },
        "configs": {str(p): sha256_file(Path(p)) for p in (config_paths or []) if Path(p).exists()},
        "artifacts": [],
        "notes": []
    }

cfg = load_yaml(CONFIGS / "run.yml")
run_cfg = cfg.get("run", {})
date_from = run_cfg.get("date_from", "2025-01-01")
date_to = run_cfg.get("date_to", "2025-01-31")
sites = cfg.get("sites", [])
sites_df = pd.DataFrame(sites)
if sites_df.empty:
    sites_df = pd.DataFrame(columns=["id","name","role","lat","lon"])

In [ ]:
step = "43_window_ranking"
out = OUTPUTS / step
out.mkdir(parents=True, exist_ok=True)

daily, dmeta = safe_read_csv("outputs/42_multi_site_comparison/daily_comparison.csv")
if daily.empty:
    raise FileNotFoundError("Run notebook 42 first.")

daily["date"] = pd.to_datetime(daily["date"], errors="coerce")
for col in ["target_minus_control","abs_difference","article_count","ground_rows"]:
    if col not in daily.columns:
        daily[col] = 0
    daily[col] = pd.to_numeric(daily[col], errors="coerce").fillna(0)

rows = []
for window_days in [4,5,6,7]:
    d = daily.copy().sort_values("date").reset_index(drop=True)
    d["abs_roll"] = d["abs_difference"].rolling(window_days, min_periods=window_days).mean()
    d["news_roll"] = d["article_count"].rolling(window_days, min_periods=1).sum()
    d["ground_roll"] = d["ground_rows"].rolling(window_days, min_periods=1).sum()
    d["window_score"] = d["abs_roll"] * 0.6 + d["news_roll"] * 0.25 + d["ground_roll"] * 0.15
    for _, r in d.iterrows():
        if pd.isna(r["abs_roll"]):
            continue
        rows.append({
            "window_days": window_days,
            "window_end": r["date"],
            "window_start": r["date"] - pd.Timedelta(days=window_days-1),
            "abs_roll": float(r["abs_roll"]),
            "news_roll": float(r["news_roll"]),
            "ground_roll": float(r["ground_roll"]),
            "window_score": float(r["window_score"]),
        })

windows = pd.DataFrame(rows).sort_values("window_score", ascending=False).reset_index(drop=True)
top = windows.head(20).copy()
windows.to_csv(out / "window_candidates.csv", index=False)
top.to_csv(out / "window_top_candidates.csv", index=False)
display(top)

fig, ax = plt.subplots(figsize=(10, 4))
n = min(20, len(top))
if n:
    ax.plot(range(1, n+1), top.head(n)["window_score"], marker="o")
ax.set_title("Top window candidate scores")
ax.set_xlabel("Rank")
ax.set_ylabel("Window score")
fig.tight_layout()
fig.savefig(out / "window_scores_plot.png", dpi=150, bbox_inches="tight")

manifest = manifest_base(step, [CONFIGS / "run.yml"])
for p in [out / "window_candidates.csv", out / "window_top_candidates.csv", out / "window_scores_plot.png"]:
    manifest["artifacts"].append({"path": str(p), "sha256": sha256_file(p)})
manifest["inputs"] = {"daily_comparison": dmeta}
write_json(out / "manifest.json", manifest)
print("Wrote:", out)